# 🚀 Alien Rescue — Game Analytics

**Analyzing 159 players · 85,194 log events · 1-hour gameplay session**

> *"What separates successful players from those who fail —  
> and how can the game be improved to help everyone succeed?"*

---

### Dataset
**Source:** Liu, S. & Liu, M. (2019). *Data on player activity and characteristics in a Serious Game Environment.*  
Data in Brief. DOI: [10.1016/j.dib.2019.104965](https://doi.org/10.1016/j.dib.2019.104965)

**Game:** Alien Rescue — players must find suitable planets for 6 displaced alien species using scientific tools.

| File | Rows | Description |
|---|---|---|
| Log_Raw | 85,194 | Every click, note, and navigation action |
| Consoles | 5,916 | Tool open/close events |
| Gates | 4,460 | Zone door crossings |
| Duration_Characteristics | 159 | Per-player summary + psychological scores |

---

### Analysis Structure

| # | Section | Business Question |
|---|---|---|
| 1 | Data Loading & Cleaning | How do we prepare 85k raw events? |
| 2 | Feature Engineering | How do we summarize each player in one row? |
| 3 | Exploratory Analysis | What does the data tell us at first glance? |
| 4 | Player Segmentation | Who are our players? Are they all the same? |
| 5 | Early Warning System | Can we detect struggling players in first 20 min? |
| 6 | Tool Analysis | Which tools create value — which need redesign? |


---
## 01 — Data Loading & Cleaning

### Why this matters
Raw game log data is messy by nature. Before any analysis, we need to:
- Standardize column names (original names had `|__dataLog__` prefixes)
- Parse timestamps from string to datetime format
- Remove duplicate player records
- Understand what each table represents

### What each file contains
- **Log_Raw**: every single player action — the richest source
- **Consoles**: which tools were opened/closed and when
- **Gates**: which game zones were entered and when  
- **Duration_Characteristics**: pre-computed per-player summaries + survey data (metacognition, goal orientation, solution score)


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

plt.rcParams['figure.figsize'] = (12, 6)
plt.rcParams['font.family'] = 'DejaVu Sans'
sns.set_theme(style="whitegrid", palette="muted")

OUTPUT_DIR = '../outputs/figures/'

# ── Load files ─────────────────────────────────────────────
# Files use tab-separation (\t), not comma
# on_bad_lines='skip' handles a small number of malformed rows

log_raw  = pd.read_csv('../data/Log_Raw.csv',   sep='\t', on_bad_lines='skip')
consoles = pd.read_csv('../data/Consoles.csv',  sep='\t', on_bad_lines='skip')
gates    = pd.read_csv('../data/Gates.csv',     sep='\t', on_bad_lines='skip')
duration = pd.read_csv('../data/Duration_Charateristics.csv')

print(f"log_raw  : {log_raw.shape[0]:,} rows × {log_raw.shape[1]} cols")
print(f"consoles : {consoles.shape[0]:,} rows × {consoles.shape[1]} cols")
print(f"gates    : {gates.shape[0]:,} rows × {gates.shape[1]} cols")
print(f"duration : {duration.shape[0]:,} rows × {duration.shape[1]} cols")


In [ ]:
# ── Rename columns ─────────────────────────────────────────
log_raw.columns  = ['action', 'note', 'timestamp', 'tool', 'user_id']
consoles.columns = ['action', 'timestamp', 'tool', 'user_id']
gates.columns    = ['action', 'note', 'timestamp', 'user_id']
duration.columns = [
    'user_id',
    'dur_alien_db', 'dur_comm_center', 'dur_concepts_db',
    'dur_mission_control', 'dur_missions_db', 'dur_notebook',
    'dur_periodic_table', 'dur_probe_design', 'dur_solar_db', 'dur_spectra',
    'gender', 'mc_average', 'solution_score',
    'tap', 'tav', 'sap', 'sav', 'oap', 'oav'
]

# ── Parse timestamps ────────────────────────────────────────
# Convert string like "Wednesday, November 29, 2017 10:35 AM" to datetime
# errors='coerce' turns unparseable values into NaT instead of raising errors

log_raw['timestamp']  = pd.to_datetime(log_raw['timestamp'],  errors='coerce')
consoles['timestamp'] = pd.to_datetime(consoles['timestamp'], errors='coerce')
gates['timestamp']    = pd.to_datetime(gates['timestamp'],    errors='coerce')

# ── Clean duration table ────────────────────────────────────
# 168 rows but only 159 unique players → remove duplicates
duration = duration.drop_duplicates(subset='user_id', keep='first')
duration = duration.dropna(subset=['solution_score'])

print(f"✓ After cleaning: {len(duration)} unique players")
print(f"✓ Timestamp NaT count in log_raw: {log_raw['timestamp'].isna().sum()}")

duration.head(3)


---
## 02 — Feature Engineering

### Why this matters
We have 85,194 rows — one per event. But to compare players and build segments,  
we need **one row per player** that summarizes their entire session.

This process is called **feature engineering**: transforming raw events into meaningful metrics.

### Features we'll create

| Feature | Source | What it captures |
|---|---|---|
| `total_actions` | log_raw | Overall engagement level |
| `notes_taken` | log_raw | Knowledge organization (metacognition proxy) |
| `section_clicks` | log_raw | Information-seeking behavior |
| `probe_actions` | log_raw | Problem-solving engagement |
| `total_gate_crossings` | gates | Map exploration |
| `unique_tools_used` | log_raw | Breadth of tool usage |
| `notebook_ratio` | derived | % of session time spent on notes |
| `action_density` | derived | Actions per minute (pace of play) |


In [ ]:
# ── From log_raw ───────────────────────────────────────────

total_actions = log_raw.groupby('user_id')['action'].count().rename('total_actions')

notes_taken = (
    log_raw[log_raw['action'].str.contains('Creat Note', na=False)]
    .groupby('user_id')['action'].count().rename('notes_taken')
)

tool_cols = log_raw.groupby('user_id')['tool'].nunique().rename('unique_tools_used')

section_clicks = (
    log_raw[log_raw['action'] == 'Click Section']
    .groupby('user_id')['action'].count().rename('section_clicks')
)

probe_actions = (
    log_raw[log_raw['tool'].str.contains('Probe|probe', na=False)]
    .groupby('user_id')['action'].count().rename('probe_actions')
)

# ── From gates ──────────────────────────────────────────────

total_gates = gates.groupby('user_id')['action'].count().rename('total_gate_crossings')

gate_types = (
    gates.groupby(['user_id', 'note'])['action']
    .count().unstack(fill_value=0)
)
gate_types.columns = [f'gate_{c.lower().replace(" ", "_")}' for c in gate_types.columns]

# ── From consoles ───────────────────────────────────────────

console_opens  = (
    consoles[consoles['action'] == 'Open']
    .groupby('user_id')['action'].count().rename('console_open_count')
)
unique_consoles = (
    consoles[consoles['action'] == 'Open']
    .groupby('user_id')['tool'].nunique().rename('unique_consoles')
)

print("✓ All features computed")


In [ ]:
# ── Merge into player_profile ───────────────────────────────

player_profile = (
    duration.set_index('user_id')
    .join(total_actions,   how='left')
    .join(notes_taken,     how='left')
    .join(tool_cols,       how='left')
    .join(section_clicks,  how='left')
    .join(probe_actions,   how='left')
    .join(total_gates,     how='left')
    .join(gate_types,      how='left')
    .join(console_opens,   how='left')
    .join(unique_consoles, how='left')
    .reset_index()
)

# Fill NaN with 0 (player never performed that action)
fill_cols = ['total_actions', 'notes_taken', 'unique_tools_used',
             'section_clicks', 'probe_actions', 'total_gate_crossings',
             'console_open_count', 'unique_consoles']
gate_cols = [c for c in player_profile.columns if c.startswith('gate_')]
player_profile[fill_cols + gate_cols] = player_profile[fill_cols + gate_cols].fillna(0)

# ── Derived features ────────────────────────────────────────

dur_cols = [c for c in player_profile.columns if c.startswith('dur_')]
player_profile['total_tool_time']  = player_profile[dur_cols].sum(axis=1)
player_profile['notebook_ratio']   = player_profile['dur_notebook'] / player_profile['total_tool_time'].replace(0, np.nan)
player_profile['action_density']   = player_profile['total_actions'] / player_profile['total_tool_time'].replace(0, np.nan)
player_profile['gender_label']     = player_profile['gender'].map({1: 'Male', 2: 'Female'})
player_profile['took_notes']       = player_profile['notes_taken'].apply(
    lambda x: 'Took Notes (≥1)' if x >= 1 else 'No Notes (0)'
)

print(f"✓ player_profile ready: {player_profile.shape[0]} players × {player_profile.shape[1]} features")
player_profile[['total_actions', 'notes_taken', 'solution_score', 'notebook_ratio']].describe().round(2)


---
## 03 — Exploratory Data Analysis

### Why this matters
Before building models, we need to understand the data visually.  
Each chart below answers a specific business question.

### Questions we'll answer
1. Are players generally succeeding? *(score distribution)*
2. Where do players spend their time? *(tool usage)*  
3. Does note-taking actually help? *(notebook vs performance)*
4. Is more activity always better? *(actions vs score)*
5. Which features correlate most with success? *(heatmap)*


In [ ]:
# ── 1. Performance Overview ────────────────────────────────

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('Player Performance Overview', fontsize=15, fontweight='bold')

score_counts = player_profile['solution_score'].value_counts().sort_index()
axes[0].bar(score_counts.index, score_counts.values,
            color=sns.color_palette("muted")[0], edgecolor='white')
axes[0].set_title('Score Distribution (0=no solution, 7=perfect)', fontsize=12)
axes[0].set_xlabel('Solution Score')
axes[0].set_ylabel('Number of Players')
for score, count in score_counts.items():
    axes[0].text(score, count + 0.3, str(count), ha='center', fontsize=9)

bins   = [(player_profile['solution_score'] == 0).sum(),
          ((player_profile['solution_score'] >= 1) & (player_profile['solution_score'] <= 3)).sum(),
          (player_profile['solution_score'] >= 4).sum()]
labels = [f'Failed\n(score=0)\n{bins[0]} players',
          f'Struggling\n(score 1-3)\n{bins[1]} players',
          f'Successful\n(score 4-7)\n{bins[2]} players']
axes[1].pie(bins, labels=labels, colors=['#e74c3c','#f39c12','#2ecc71'],
            autopct='%1.0f%%', startangle=90, pctdistance=0.6)
axes[1].set_title('Player Success Segments', fontsize=12)

plt.tight_layout()
plt.savefig(OUTPUT_DIR + '01_performance_overview.png', dpi=150, bbox_inches='tight')
plt.show()

print("\n🔍 Key finding: 33% of players scored 0 despite completing a full 1-hour session.")
print("   This points to a critical onboarding or discoverability problem in the game.")


In [ ]:
# ── 2. Tool Usage ──────────────────────────────────────────

tool_name_map = {
    'dur_alien_db':'Alien DB', 'dur_comm_center':'Comm Center',
    'dur_concepts_db':'Concepts DB', 'dur_mission_control':'Mission Control',
    'dur_missions_db':'Missions DB', 'dur_notebook':'Notebook',
    'dur_periodic_table':'Periodic Table', 'dur_probe_design':'Probe Design',
    'dur_solar_db':'Solar DB', 'dur_spectra':'Spectra'
}

tool_means = player_profile[list(tool_name_map.keys())].mean().rename(tool_name_map).sort_values(ascending=True)

fig, axes = plt.subplots(1, 2, figsize=(16, 6))
fig.suptitle('Tool Usage Analysis', fontsize=15, fontweight='bold')

bars = axes[0].barh(tool_means.index, tool_means.values,
                    color=sns.color_palette("muted", len(tool_means)))
axes[0].set_title('Average Time Spent per Tool (minutes)', fontsize=12)
axes[0].set_xlabel('Average Duration (minutes)')
for bar, val in zip(bars, tool_means.values):
    axes[0].text(val + 0.3, bar.get_y() + bar.get_height()/2,
                 f'{val:.1f}', va='center', fontsize=9)

tool_data_long = player_profile[list(tool_name_map.keys())].rename(columns=tool_name_map).melt(
    var_name='Tool', value_name='Duration')
tool_order = tool_means.index.tolist()
axes[1].boxplot(
    [tool_data_long[tool_data_long['Tool'] == t]['Duration'].values for t in tool_order],
    labels=tool_order, vert=False, patch_artist=True,
    boxprops=dict(facecolor='#a8d8ea', alpha=0.7))
axes[1].set_title('Duration Distribution per Tool', fontsize=12)
axes[1].set_xlabel('Duration (minutes)')

plt.tight_layout()
plt.savefig(OUTPUT_DIR + '02_tool_usage.png', dpi=150, bbox_inches='tight')
plt.show()

print("\n🔍 Key finding: Alien DB dominates time (avg 20 min) but has near-zero score correlation.")
print("   Probe Design — the core problem-solving tool — receives far less time than expected.")


In [ ]:
# ── 3. Notebook vs Performance ─────────────────────────────

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('Notebook Usage vs Performance', fontsize=15, fontweight='bold')

scatter = axes[0].scatter(player_profile['notes_taken'], player_profile['solution_score'],
    c=player_profile['solution_score'], cmap='RdYlGn', alpha=0.7, s=60,
    edgecolors='white', linewidth=0.5)
plt.colorbar(scatter, ax=axes[0], label='Solution Score')
z = np.polyfit(player_profile['notes_taken'], player_profile['solution_score'], 1)
x_line = np.linspace(player_profile['notes_taken'].min(), player_profile['notes_taken'].max(), 100)
axes[0].plot(x_line, np.poly1d(z)(x_line), 'navy', linestyle='--', linewidth=1.5, label='Trend')
axes[0].legend()
corr = player_profile['notes_taken'].corr(player_profile['solution_score'])
axes[0].text(0.05, 0.92, f'r = {corr:.2f}', transform=axes[0].transAxes,
             fontsize=11, color='navy', fontweight='bold')
axes[0].set_xlabel('Number of Notes Taken')
axes[0].set_ylabel('Solution Score')
axes[0].set_title('Notes Taken vs Solution Score', fontsize=12)

note_group_scores = player_profile.groupby('took_notes')['solution_score'].mean()
bars2 = axes[1].bar(note_group_scores.index, note_group_scores.values,
    color=['#e74c3c','#2ecc71'], edgecolor='white', width=0.5)
axes[1].set_title('Avg Score: Note Takers vs Non-Note Takers', fontsize=12)
axes[1].set_ylabel('Average Solution Score')
axes[1].set_ylim(0, 7)
for bar, val in zip(bars2, note_group_scores.values):
    count = (player_profile['took_notes'] == bar.get_x()).sum()
    axes[1].text(bar.get_x() + bar.get_width()/2, val + 0.1,
                 f'{val:.2f}', ha='center', fontsize=12, fontweight='bold')

plt.tight_layout()
plt.savefig(OUTPUT_DIR + '03_notebook_vs_performance.png', dpi=150, bbox_inches='tight')
plt.show()

print(f"\n🔍 Key finding: Note-takers average {note_group_scores.max():.2f} vs {note_group_scores.min():.2f} for non-note-takers.")
print("   The Notebook feature works — but most players don't discover or use it.")


In [ ]:
# ── 4 & 5. Activity + Correlation Heatmap ──────────────────

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('Player Activity vs Performance', fontsize=15, fontweight='bold')

scatter2 = axes[0].scatter(player_profile['total_actions'], player_profile['solution_score'],
    c=player_profile['mc_average'], cmap='coolwarm', alpha=0.7, s=60,
    edgecolors='white', linewidth=0.5)
plt.colorbar(scatter2, ax=axes[0], label='Metacognition Score')
z2 = np.polyfit(player_profile['total_actions'], player_profile['solution_score'], 1)
x2 = np.linspace(player_profile['total_actions'].min(), player_profile['total_actions'].max(), 100)
axes[0].plot(x2, np.poly1d(z2)(x2), 'navy', linestyle='--', linewidth=1.5)
corr2 = player_profile['total_actions'].corr(player_profile['solution_score'])
axes[0].text(0.05, 0.92, f'r = {corr2:.2f}', transform=axes[0].transAxes,
             fontsize=11, color='navy', fontweight='bold')
axes[0].set_xlabel('Total Actions')
axes[0].set_ylabel('Solution Score')
axes[0].set_title('Total Actions vs Score\n(color = metacognition)', fontsize=12)

scatter3 = axes[1].scatter(player_profile['mc_average'], player_profile['solution_score'],
    c=player_profile['total_actions'], cmap='YlOrRd', alpha=0.7, s=60,
    edgecolors='white', linewidth=0.5)
plt.colorbar(scatter3, ax=axes[1], label='Total Actions')
corr3 = player_profile['mc_average'].corr(player_profile['solution_score'])
axes[1].text(0.05, 0.92, f'r = {corr3:.2f}', transform=axes[1].transAxes,
             fontsize=11, color='navy', fontweight='bold')
axes[1].set_xlabel('Metacognition Score')
axes[1].set_ylabel('Solution Score')
axes[1].set_title('Metacognition vs Score\n(color = total actions)', fontsize=12)

plt.tight_layout()
plt.savefig(OUTPUT_DIR + '04_activity_vs_performance.png', dpi=150, bbox_inches='tight')
plt.show()

print("\n🔍 Key finding: More actions ≠ better score (r = {:.2f}).".format(corr2))
print("   High metacognition doesn't guarantee success — explored further in segmentation.")


In [ ]:
# ── Correlation Heatmap ────────────────────────────────────

corr_cols = ['solution_score', 'mc_average', 'total_actions', 'notes_taken',
             'section_clicks', 'probe_actions', 'total_gate_crossings',
             'console_open_count', 'unique_tools_used', 'total_tool_time',
             'notebook_ratio', 'action_density', 'dur_probe_design',
             'dur_notebook', 'dur_alien_db']

corr_matrix = player_profile[corr_cols].corr()
mask = np.triu(np.ones_like(corr_matrix, dtype=bool))

fig, ax = plt.subplots(figsize=(13, 10))
sns.heatmap(corr_matrix, mask=mask, annot=True, fmt='.2f', cmap='RdBu_r',
            center=0, vmin=-1, vmax=1, ax=ax, annot_kws={'size': 8}, linewidths=0.5)
ax.set_title('Feature Correlation Matrix\n(what drives solution_score?)',
             fontsize=13, fontweight='bold', pad=15)
plt.xticks(rotation=45, ha='right', fontsize=9)
plt.yticks(rotation=0, fontsize=9)
plt.tight_layout()
plt.savefig(OUTPUT_DIR + '05_correlation_heatmap.png', dpi=150, bbox_inches='tight')
plt.show()

print("\nTop correlations with solution_score:")
corr_with_score = corr_matrix['solution_score'].drop('solution_score').sort_values(key=abs, ascending=False)
for feat, val in corr_with_score.head(6).items():
    print(f"  {'↑' if val > 0 else '↓'} {feat:<25} r = {val:.3f}")


---
## 04 — Player Segmentation (K-Means Clustering)

### Why this matters
Not all players are the same. Treating them as one group leads to one-size-fits-all design decisions  
that work poorly for everyone. Segmentation lets us ask: **who are our players, really?**

### Method: K-Means Clustering
K-Means groups players by minimizing the distance between each player and their cluster center.  
We use **10 behavioral features** (no psychological scores — pure in-game behavior).

**Why StandardScaler first?**  
`total_actions = 750` and `notebook_ratio = 0.07` can't be compared directly.  
Scaling transforms every feature to: *(value − mean) / std deviation*  
Now all features live on the same scale and K-Means distances are meaningful.

**How we chose k=3:**  
- Elbow method: inertia reduction slows around k=3
- Silhouette score: measures cluster separation quality
- Domain knowledge: 3 segments ("good", "medium", "lost") makes business sense


In [ ]:
from sklearn.preprocessing import StandardScaler, MinMaxScaler
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA
from sklearn.metrics import silhouette_score

segment_colors = {'Achievers': '#2ecc71', 'Explorers': '#3498db', 'Lost Players': '#e74c3c'}
seg_order = ['Lost Players', 'Explorers', 'Achievers']

cluster_features = [
    'total_actions', 'notes_taken', 'section_clicks', 'probe_actions',
    'total_gate_crossings', 'unique_tools_used', 'dur_notebook',
    'dur_probe_design', 'notebook_ratio', 'action_density'
]

cluster_data = player_profile[cluster_features].dropna()
valid_idx    = cluster_data.index
X_scaled     = StandardScaler().fit_transform(cluster_data)

# ── Find optimal k ──────────────────────────────────────────

inertias, silhouette_scores_list = [], []
k_range = range(2, 8)

for k in k_range:
    km = KMeans(n_clusters=k, random_state=42, n_init=10)
    km.fit(X_scaled)
    inertias.append(km.inertia_)
    silhouette_scores_list.append(silhouette_score(X_scaled, km.labels_))

fig, axes = plt.subplots(1, 2, figsize=(13, 5))
fig.suptitle('Finding Optimal Number of Segments (k)', fontsize=14, fontweight='bold')

axes[0].plot(k_range, inertias, 'o-', color='steelblue', linewidth=2, markersize=8)
axes[0].set_xlabel('Number of Clusters (k)')
axes[0].set_ylabel('Inertia')
axes[0].set_title('Elbow Method')
axes[0].set_xticks(list(k_range))

best_k = list(k_range)[silhouette_scores_list.index(max(silhouette_scores_list))]
axes[1].plot(k_range, silhouette_scores_list, 's-', color='coral', linewidth=2, markersize=8)
axes[1].axvline(x=best_k, color='red', linestyle='--', alpha=0.7)
axes[1].set_xlabel('Number of Clusters (k)')
axes[1].set_ylabel('Silhouette Score (higher = better)')
axes[1].set_title('Silhouette Score')
axes[1].set_xticks(list(k_range))

plt.tight_layout()
plt.savefig(OUTPUT_DIR + '06_optimal_k.png', dpi=150, bbox_inches='tight')
plt.show()

print(f"Silhouette best k = {best_k} | We use k=3 for business interpretability")


In [ ]:
# ── Fit K-Means k=3 ────────────────────────────────────────

kmeans = KMeans(n_clusters=3, random_state=42, n_init=10)
labels = kmeans.fit_predict(X_scaled)

player_profile['segment'] = np.nan
player_profile.loc[valid_idx, 'segment'] = labels
player_profile['segment'] = player_profile['segment'].astype('Int64')

# Label segments by performance rank
seg_scores  = player_profile.groupby('segment')['solution_score'].mean()
score_rank  = seg_scores.rank()
segment_labels = {seg: ('Achievers' if r==3 else 'Explorers' if r==2 else 'Lost Players')
                  for seg, r in score_rank.items()}
player_profile['segment_name'] = player_profile['segment'].map(segment_labels)

print("Segment summary:")
print(player_profile.groupby('segment_name').agg(
    count=('user_id','count'),
    avg_score=('solution_score','mean'),
    avg_actions=('total_actions','mean'),
    avg_notes=('notes_taken','mean')
).round(2).reindex(seg_order))


In [ ]:
# ── Visualize segments ─────────────────────────────────────

pca = PCA(n_components=2, random_state=42)
X_pca = pca.fit_transform(X_scaled)
explained = pca.explained_variance_ratio_

fig, axes = plt.subplots(1, 2, figsize=(16, 6))
fig.suptitle('Player Segmentation Results', fontsize=15, fontweight='bold')

for seg_name, color in segment_colors.items():
    mask = player_profile.loc[valid_idx, 'segment_name'] == seg_name
    axes[0].scatter(X_pca[mask, 0], X_pca[mask, 1], c=color,
        label=f'{seg_name} (n={mask.sum()})', alpha=0.75, s=70,
        edgecolors='white', linewidth=0.5)
axes[0].set_xlabel(f'PC1 ({explained[0]*100:.1f}% variance)')
axes[0].set_ylabel(f'PC2 ({explained[1]*100:.1f}% variance)')
axes[0].set_title('Player Segments in PCA Space', fontsize=12)
axes[0].legend(fontsize=10)

seg_perf  = player_profile.groupby('segment_name')['solution_score'].agg(['mean','std']).reindex(seg_order)
bars = axes[1].bar(seg_perf.index, seg_perf['mean'],
    color=[segment_colors[s] for s in seg_order], edgecolor='white', width=0.5,
    yerr=seg_perf['std'], capsize=6, error_kw={'linewidth':1.5})
axes[1].set_title('Average Solution Score by Segment', fontsize=12)
axes[1].set_ylabel('Average Solution Score (0-7)')
axes[1].set_ylim(0, 8)
for bar, (seg, row) in zip(bars, seg_perf.iterrows()):
    count = (player_profile['segment_name'] == seg).sum()
    axes[1].text(bar.get_x() + bar.get_width()/2,
                 row['mean'] + row['std'] + 0.2,
                 f"{row['mean']:.1f}\n(n={count})",
                 ha='center', fontsize=10, fontweight='bold')

plt.tight_layout()
plt.savefig(OUTPUT_DIR + '07_segments_overview.png', dpi=150, bbox_inches='tight')
plt.show()

print("\n🔍 Key finding: Lost Players generate the MOST actions (748 avg) but score lowest.")
print("   Activity volume ≠ strategic engagement. The game rewards clicking, not thinking.")


In [ ]:
# ── Radar chart + Tool heatmap ─────────────────────────────

radar_features = ['total_actions','notes_taken','section_clicks',
                  'probe_actions','total_gate_crossings','notebook_ratio']
radar_labels   = ['Total\nActions','Notes\nTaken','Section\nClicks',
                  'Probe\nActions','Gate\nCrossings','Notebook\nRatio']

seg_radar      = player_profile.groupby('segment_name')[radar_features].mean()
seg_radar_norm = pd.DataFrame(MinMaxScaler().fit_transform(seg_radar),
                               index=seg_radar.index, columns=radar_features)

N      = len(radar_features)
angles = [n / float(N) * 2 * np.pi for n in range(N)]
angles += angles[:1]

fig, ax = plt.subplots(figsize=(8, 8), subplot_kw=dict(polar=True))
for seg_name in seg_order:
    if seg_name not in seg_radar_norm.index: continue
    values = seg_radar_norm.loc[seg_name].tolist() + [seg_radar_norm.loc[seg_name].tolist()[0]]
    ax.plot(angles, values, linewidth=2.5, label=seg_name, color=segment_colors[seg_name])
    ax.fill(angles, values, alpha=0.12, color=segment_colors[seg_name])

ax.set_xticks(angles[:-1])
ax.set_xticklabels(radar_labels, size=11)
ax.set_ylim(0, 1)
ax.set_title('Behavioral Fingerprint by Segment', size=14, fontweight='bold', pad=25)
ax.legend(loc='upper right', bbox_to_anchor=(1.3, 1.15), fontsize=11)
plt.tight_layout()
plt.savefig(OUTPUT_DIR + '08_segment_radar.png', dpi=150, bbox_inches='tight')
plt.show()


---
## 05 — Early Warning System

### Why this matters
If we can identify struggling players **before they fail**, we can intervene:  
show a hint, redirect them to a key tool, or trigger a tutorial prompt.

### Approach
1. Isolate each player's **first 20 minutes** of gameplay
2. Extract 7 behavioral signals from that window
3. Test whether those signals predict final score (Random Forest)
4. Build a **risk score** as an actionable output

### Key result (spoiler)
The first 20 minutes **cannot reliably predict** final score alone (R² ≈ -0.07).  
This is itself an important finding — the critical decision point happens **mid-game**, not at the start.  
However, a composite risk score based on 4 early signals still meaningfully separates player groups.


In [ ]:
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import cross_val_score

WINDOW = 20  # minutes

log_timed = log_raw.merge(
    log_raw.groupby('user_id')['timestamp'].min().rename('session_start'),
    on='user_id'
)
log_timed['minutes_elapsed'] = (
    (log_timed['timestamp'] - log_timed['session_start']).dt.total_seconds() / 60
)

early_log = log_timed[log_timed['minutes_elapsed'] <= WINDOW].copy()

early_features = (
    early_log.groupby('user_id').agg(
        early_actions       = ('action', 'count'),
        early_unique_tools  = ('tool', 'nunique'),
        early_notes         = ('action', lambda x: x.str.contains('Creat Note', na=False).sum()),
        early_probe         = ('tool', lambda x: x.str.contains('Probe|probe', na=False).sum()),
        early_section_clicks= ('action', lambda x: (x == 'Click Section').sum()),
    ).reset_index()
)
early_features['tried_probe_early'] = (early_features['early_probe'] > 0).astype(int)
early_features['took_notes_early']  = (early_features['early_notes'] > 0).astype(int)

early_analysis = early_features.merge(
    player_profile[['user_id','solution_score','segment_name']], on='user_id', how='inner'
)

early_feat_cols = ['early_actions','early_unique_tools','early_notes',
                   'early_probe','early_section_clicks','tried_probe_early','took_notes_early']
rf = RandomForestRegressor(n_estimators=100, random_state=42)
cv_scores = cross_val_score(rf, early_analysis[early_feat_cols],
                             early_analysis['solution_score'], cv=5, scoring='r2')

print(f"Random Forest R² (5-fold CV): {cv_scores.mean():.3f} ± {cv_scores.std():.3f}")
print(f"\nInterpretation: First {WINDOW} minutes explain only {max(cv_scores.mean()*100, 0):.1f}% of score variance.")
print("The critical engagement window is mid-game, not the opening.")


In [ ]:
# ── Feature importance + Risk score ────────────────────────

rf.fit(early_analysis[early_feat_cols], early_analysis['solution_score'])
importances = pd.Series(rf.feature_importances_, index=early_feat_cols).sort_values(ascending=True)

low_action_threshold = early_analysis['early_actions'].quantile(0.25)
low_tool_threshold   = early_analysis['early_unique_tools'].quantile(0.25)

early_analysis['risk_low_activity'] = (early_analysis['early_actions'] < low_action_threshold).astype(int)
early_analysis['risk_no_notes']     = (early_analysis['took_notes_early'] == 0).astype(int)
early_analysis['risk_no_probe']     = (early_analysis['tried_probe_early'] == 0).astype(int)
early_analysis['risk_low_tools']    = (early_analysis['early_unique_tools'] < low_tool_threshold).astype(int)
early_analysis['risk_score']        = early_analysis[
    ['risk_low_activity','risk_no_notes','risk_no_probe','risk_low_tools']
].sum(axis=1) / 4 * 100
early_analysis['risk_group'] = pd.cut(early_analysis['risk_score'],
    bins=[-1,25,50,75,101], labels=['Low Risk','Medium Risk','High Risk','Critical Risk'])

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('Player Risk Score — Early Warning System', fontsize=14, fontweight='bold')

risk_colors = ['#2ecc71','#f1c40f','#e67e22','#e74c3c']
risk_order  = ['Low Risk','Medium Risk','High Risk','Critical Risk']
risk_counts = early_analysis['risk_group'].value_counts().reindex(risk_order)
risk_perf   = early_analysis.groupby('risk_group', observed=True)['solution_score'].mean().reindex(risk_order)

axes[0].bar(risk_counts.index, risk_counts.values, color=risk_colors, edgecolor='white', width=0.5)
axes[0].set_title('Risk Group Distribution', fontsize=12)
axes[0].set_ylabel('Number of Players')

axes[1].bar(risk_perf.index, risk_perf.values, color=risk_colors, edgecolor='white', width=0.5)
axes[1].set_title('Avg Final Score by Risk Group', fontsize=12)
axes[1].set_ylabel('Average Solution Score (0-7)')
axes[1].set_ylim(0, 7)
for ax, vals in zip([axes[0], axes[1]], [risk_counts, risk_perf]):
    for i, (idx, val) in enumerate(vals.items()):
        ax.text(i, val + 0.1, f'{val:.0f}' if ax == axes[0] else f'{val:.2f}',
                ha='center', fontsize=11, fontweight='bold')

plt.tight_layout()
plt.savefig(OUTPUT_DIR + '11_risk_scores.png', dpi=150, bbox_inches='tight')
plt.show()

print("\n🔍 Low Risk players average {:.2f} vs Critical Risk {:.2f} — a {:.0f}% difference.".format(
    risk_perf['Low Risk'], risk_perf['Critical Risk'],
    (risk_perf['Low Risk'] - risk_perf['Critical Risk']) / risk_perf['Critical Risk'] * 100))


---
## 06 — Tool Analysis & Design Recommendations

### Why this matters
The game has 10 tools. Not all of them are equal:
- Some are used by almost everyone but don't help (wasted screen real estate)
- Some are rarely discovered but strongly predict success (hidden gems)
- Some need to be redesigned entirely

### Tool Positioning Matrix
We evaluate each tool on two axes:
- **X axis:** Adoption rate — what % of players used it for ≥1 minute?
- **Y axis:** Value — correlation with solution score
- **Bubble size:** Average time spent

This creates 4 quadrants: Core Tools · Hidden Gems · Busy Tools · Dead Weight


In [ ]:
# ── Adoption + correlation per tool ────────────────────────

dur_cols_named = {
    'dur_alien_db':'Alien DB', 'dur_comm_center':'Comm Center',
    'dur_concepts_db':'Concepts DB', 'dur_mission_control':'Mission Control',
    'dur_missions_db':'Missions DB', 'dur_notebook':'Notebook',
    'dur_periodic_table':'Periodic Table', 'dur_probe_design':'Probe Design',
    'dur_solar_db':'Solar DB', 'dur_spectra':'Spectra'
}

tool_stats = {}
for col, name in dur_cols_named.items():
    tool_stats[name] = {
        'avg_time':         player_profile[col].mean(),
        'corr_with_score':  player_profile[col].corr(player_profile['solution_score']),
        'adoption_rate':    (player_profile[col] >= 1).mean() * 100
    }
tool_stats_df = pd.DataFrame(tool_stats).T.round(3)
mid_x = tool_stats_df['adoption_rate'].mean()

fig, ax = plt.subplots(figsize=(11, 8))
colors_tool = sns.color_palette("tab10", len(tool_stats_df))

for i, (tool_name, row) in enumerate(tool_stats_df.iterrows()):
    ax.scatter(row['adoption_rate'], row['corr_with_score'],
               s=row['avg_time'] * 25 + 80, color=colors_tool[i],
               alpha=0.8, edgecolors='white', linewidth=1.5, zorder=3)
    ax.annotate(tool_name, (row['adoption_rate'], row['corr_with_score']),
                textcoords='offset points', xytext=(8, 4), fontsize=9, fontweight='bold')

ax.axvline(x=mid_x, color='gray', linestyle='--', alpha=0.5)
ax.axhline(y=0,     color='gray', linestyle='--', alpha=0.5)

ymax = tool_stats_df['corr_with_score'].max()
ymin = tool_stats_df['corr_with_score'].min()
ax.text(mid_x+0.5, ymax*0.85, '⭐ CORE TOOLS\n(keep & enhance)',
        fontsize=9, color='#27ae60', fontweight='bold', alpha=0.8)
ax.text(tool_stats_df['adoption_rate'].min(), ymax*0.85, '💎 HIDDEN GEMS\n(promote early)',
        fontsize=9, color='#2980b9', fontweight='bold', alpha=0.8)
ax.text(mid_x+0.5, ymin*0.85, '⚠️ BUSY TOOLS\n(simplify UX)',
        fontsize=9, color='#e67e22', fontweight='bold', alpha=0.8)
ax.text(tool_stats_df['adoption_rate'].min(), ymin*0.85, '🔴 DEAD WEIGHT\n(redesign)',
        fontsize=9, color='#e74c3c', fontweight='bold', alpha=0.8)

ax.set_xlabel('Adoption Rate (% players who used ≥1 min)', fontsize=11)
ax.set_ylabel('Correlation with Solution Score', fontsize=11)
ax.set_title('Tool Positioning Matrix\n(bubble size = avg time spent)',
             fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig(OUTPUT_DIR + '12_tool_positioning_matrix.png', dpi=150, bbox_inches='tight')
plt.show()


In [ ]:
# ── Design recommendations summary ─────────────────────────

print("=" * 65)
print("TOOL ANALYSIS — DESIGN RECOMMENDATIONS")
print("=" * 65)

for tool_name, row in tool_stats_df.sort_values('corr_with_score', ascending=False).iterrows():
    a, c = row['adoption_rate'], row['corr_with_score']
    if   a >= mid_x and c >= 0: tag = "⭐ CORE TOOL    → Keep & surface early"
    elif a < mid_x  and c >= 0: tag = "💎 HIDDEN GEM   → Add to onboarding"
    elif a >= mid_x and c < 0:  tag = "⚠️  BUSY TOOL    → Simplify content"
    else:                        tag = "🔴 DEAD WEIGHT  → Redesign"
    print(f"  {tool_name:<18} | adoption: {a:5.1f}% | corr: {c:+.3f} | {tag}")


---
## 07 — Conclusions & Key Takeaways

### What the data tells us

| # | Finding | Business Implication |
|---|---|---|
| 1 | **33% of players score 0** despite a full session | Onboarding and tool discoverability must be redesigned |
| 2 | **Note-taking is the strongest behavioral predictor** | Notebook must become a first-class mechanic, not hidden |
| 3 | **Lost Players are busiest but least strategic** | A hint system triggered by unfocused behavior can help |
| 4 | **First 20 minutes don't predict final score** | Critical engagement window is mid-game |
| 5 | **Missions DB has negative score correlation** | Review content — players may be spending time unproductively |
| 6 | **Probe Design is underused despite high value** | Surface it in the first 5 minutes of gameplay |

---

### Limitations
- Sample size is n=159 — findings are exploratory, not statistically definitive
- All players are undergraduate students from one university — limited generalizability
- Correlation ≠ causation — further A/B testing needed to validate design recommendations

---

### Next Steps
1. **A/B test** notebook tutorial vs no tutorial → measure score impact
2. **Implement risk score** in real-time game telemetry → trigger hints at the 20-min mark
3. **Restructure Missions DB** content based on what high-scorers actually use it for
4. **Expand dataset** with more diverse player populations

---

*Dataset: Liu & Liu (2019) — University of Texas at Austin, Alien Rescue Research Team*  
*Analysis: Python · pandas · scikit-learn · matplotlib · seaborn*
